# Module 5.3 — Executors (concurrent.futures)

### Python Mastery · Track 5: Concurrency & Asynchronous Programming

Managing threads and processes by hand works, but `concurrent.futures` offers a cleaner, unified interface. You submit work to an **executor**, receive a `Future` representing the eventual result, and collect results as they complete. The same code switches between threads and processes by changing one class. This module covers both executors and the `Future` interface.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Thread-based examples run directly in the notebook. Process-based examples are written to a `.py` file and run with `!python`, because worker processes cannot reliably import functions defined inside a notebook.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Submit work to a `ThreadPoolExecutor` and collect results.
2. Use `executor.map` for the common case of applying a function to many inputs.
3. Work with `Future` objects, including `result()` and `as_completed`.
4. Handle exceptions raised inside submitted tasks.
5. Switch to a `ProcessPoolExecutor` for CPU-bound work.

**Prerequisites:** Tracks 1 to 4, and Modules 5.1 to 5.2.

---

## Part 1 · The Executor Idea and `submit`

An executor manages a pool of workers for you. You call `submit(func, *args)` to schedule a call; it returns immediately with a `Future`, a handle to a result that may not exist yet. Calling `.result()` on the future waits for the work to finish and returns its value (or re-raises its exception). Using the executor as a context manager ensures it shuts down cleanly.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def slow_double(n):
    time.sleep(0.02)             # simulate input/output work
    return n * 2

with ThreadPoolExecutor(max_workers=3) as executor:
    future = executor.submit(slow_double, 10)   # schedule the call, returns at once
    print("is it done yet?", future.done())      # likely False immediately
    print("result (waits):", future.result())    # blocks until ready, then returns 20

Submitting several tasks lets them run concurrently. You can keep the futures in a list and read their results afterwards. Reading `.result()` in submission order gives results in that order.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch(item):
    time.sleep(0.02)
    return f"data for {item}"

items = ["a", "b", "c", "d"]
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(fetch, item) for item in items]   # all scheduled
    results = [f.result() for f in futures]                       # collected in order
print(results)

## Part 2 · `executor.map` for the Common Case

When you simply want to apply one function to many inputs, `executor.map` is the cleanest tool. It works like the built-in `map` but runs the calls concurrently, and it yields results in the **order of the inputs**, not the order of completion.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def process(n):
    time.sleep(0.02)
    return n * n

numbers = [1, 2, 3, 4, 5]
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(process, numbers))   # concurrent, results in input order
print(results)

## Part 3 · `as_completed` — Results as They Finish

Sometimes you want to react to each result the moment it is ready, rather than waiting for all of them in order. `as_completed` yields futures as each one finishes, so faster tasks are handled first. Mapping each future back to its input (via a dictionary) lets you tell which result is which.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def work(name, duration):
    time.sleep(duration)
    return f"{name} took {duration}s"

jobs = {"slow": 0.05, "fast": 0.01, "medium": 0.03}

finished_order = []
with ThreadPoolExecutor(max_workers=3) as executor:
    # Map each future to the job name so we can identify completions.
    future_to_name = {executor.submit(work, name, d): name for name, d in jobs.items()}
    for future in as_completed(future_to_name):
        finished_order.append(future_to_name[future])

print("completion order (fastest first):", finished_order)

## Part 4 · Handling Exceptions in Tasks

If a submitted task raises an exception, it is stored in the future and re-raised when you call `.result()`. This means errors are not lost; you handle them where you read the result, with an ordinary `try`/`except`. The other tasks are unaffected.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def risky(n):
    if n == 0:
        raise ValueError("cannot process zero")
    return 100 // n

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(risky, n): n for n in [4, 0, 2]}
    for future, n in futures.items():
        try:
            print(f"risky({n}) = {future.result()}")
        except ValueError as e:
            print(f"risky({n}) failed: {e}")

## Part 5 · Switching to Processes for CPU-Bound Work

The strength of `concurrent.futures` is that the same interface drives both threads and processes: replace `ThreadPoolExecutor` with `ProcessPoolExecutor` and the rest of the code is unchanged. Use threads for input/output-bound work and processes for CPU-bound work, for the reasons covered in Modules 5.1 and 5.2.

Because process workers must import their target function, the process example below is written to a script and run with `!python`, the same reliable approach used in Module 5.2.

In [ ]:
%%writefile pool_executor.py
from concurrent.futures import ProcessPoolExecutor

def heavy(n):
    # CPU-bound stand-in.
    return sum(i * i for i in range(n))

if __name__ == "__main__":                  # required when using processes
    inputs = [100_000, 200_000, 300_000]
    with ProcessPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(heavy, inputs))   # identical interface to threads
    for n, r in zip(inputs, results):
        print(f"heavy({n}) = {r}")

In [ ]:
!python pool_executor.py

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Submitting a single task (Easy)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def add(a, b):
    return a + b

with ThreadPoolExecutor() as executor:
    future = executor.submit(add, 3, 4)
    print("result:", future.result())

### Example 2 — map over inputs (Easy)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def to_upper(text):
    return text.upper()

with ThreadPoolExecutor(max_workers=3) as executor:
    print(list(executor.map(to_upper, ["a", "b", "c"])))

### Example 3 — Concurrent input/output with a speed comparison (Medium)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch(url):
    time.sleep(0.03)                 # simulate a network call
    return f"content of {url}"

urls = [f"site{i}.com" for i in range(6)]

start = time.perf_counter()
serial = [fetch(u) for u in urls]
serial_time = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=6) as executor:
    concurrent = list(executor.map(fetch, urls))
concurrent_time = time.perf_counter() - start

print(f"serial:     {serial_time:.2f}s")
print(f"concurrent: {concurrent_time:.2f}s")
print("same results:", serial == concurrent)

### Example 4 — Reacting to results with as_completed (Medium)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def compute(n):
    time.sleep(0.04 - n * 0.01)      # larger n finishes sooner
    return n, n ** 2

order = []
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(compute, n) for n in [1, 2, 3]]
    for future in as_completed(futures):
        n, squared = future.result()
        order.append(n)
        print(f"{n} squared is {squared}")

print("order of completion:", order)

### Example 5 — Collecting both successes and failures (Difficult)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def parse(value):
    return int(value)                # raises ValueError on bad input

inputs = ["10", "x", "20", "y", "30"]
successes, failures = [], []

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(parse, v): v for v in inputs}
    for future in as_completed(futures):
        value = futures[future]
        try:
            successes.append(future.result())
        except ValueError:
            failures.append(value)

print("parsed:", sorted(successes))
print("failed:", sorted(failures))

### Example 6 — A bounded pipeline that aggregates results (Difficult)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def download(item):
    time.sleep(0.01)
    return len(item)                 # pretend: the size of the downloaded item

def total_size(items, workers):
    with ThreadPoolExecutor(max_workers=workers) as executor:
        sizes = list(executor.map(download, items))   # concurrent downloads
    return sum(sizes)

files = ["a.txt", "report.pdf", "image.png", "notes.md"]
print("total size:", total_size(files, workers=4))

---

## Exercises

**Exercise 1 (Easy).** Use a `ThreadPoolExecutor` to submit a single call to a function `square(n)` and print the result via the future.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Use `executor.map` with a `ThreadPoolExecutor` to apply a function `triple(n)` to the numbers 1 to 5 and print the list of results.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Submit five tasks that each return their input doubled, then collect the results in submission order from a list of futures.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Use `as_completed` to process five tasks and print each result as it finishes, mapping each future back to its input number.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Submit a mix of tasks where some raise an exception (for example, dividing 100 by each of `[5, 0, 4, 0, 2]`). Collect successful results and a count of failures, handling the exception where you read each result.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def square(n):
    return n * n

with ThreadPoolExecutor() as executor:
    print(executor.submit(square, 6).result())

**Solution 2**

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def triple(n):
    return n * 3

with ThreadPoolExecutor(max_workers=3) as executor:
    print(list(executor.map(triple, range(1, 6))))

**Solution 3**

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def double(n):
    return n * 2

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(double, n) for n in range(5)]
    print([f.result() for f in futures])

**Solution 4**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def work(n):
    return n, n + 100

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(work, n): n for n in range(5)}
    for future in as_completed(futures):
        n, value = future.result()
        print(f"{n} -> {value}")

**Solution 5**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def divide(n):
    return 100 / n

results, failures = [], 0
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(divide, n) for n in [5, 0, 4, 0, 2]]
    for future in as_completed(futures):
        try:
            results.append(future.result())
        except ZeroDivisionError:
            failures += 1

print("results:", sorted(results), "| failures:", failures)

---

## Summary and Key Points

- An executor manages a pool of workers; `submit(func, *args)` returns a `Future` immediately, and `future.result()` waits for the value or re-raises the task's exception.
- `executor.map` applies a function to many inputs concurrently and yields results in input order, ideal for the common case.
- `as_completed` yields futures as each finishes, letting you react to fastest-first results; map futures to inputs to identify them.
- Exceptions raised in a task are stored in its future and re-raised at `.result()`, so handle them with an ordinary `try`/`except` where you read results; other tasks continue.
- The same interface drives both kinds of work: use `ThreadPoolExecutor` for input/output-bound tasks and `ProcessPoolExecutor` for CPU-bound tasks, changing only the class.

### Next module: 5.4 — Asyncio Foundations

The next module introduces asynchronous programming with `async` and `await`: coroutines, the event loop, tasks, and concurrent waiting, with attention to how this runs inside a notebook.